# LangChain Weather Agent with Full Chat-History Memory and LangSmith Tracing

This beginner-friendly Colab notebook builds a **weather agent** using LangChain, OpenRouter, and OpenWeatherMap, then traces its execution with **LangSmith**.

We will first understand and build a basic weather agent. Then we will add **full chat-history memory**, allowing the agent to understand follow-up questions such as:

- `What is the weather in Kolkata?`
- `What is the humidity there?`
- `Based on that, should I carry an umbrella?`

## Learning objectives

By the end of this notebook, you will be able to:

1. Explain what a weather agent does.
2. Create a LangChain tool that calls a live weather API.
3. Give the tool to an LLM-based agent.
4. Store and resend the complete conversation history.
5. Ask follow-up questions that depend on earlier messages.
6. View the agent, LLM, and weather-tool execution in LangSmith.

## 1. Install the required packages

- `langchain` provides the agent and tool framework.
- `langchain-openrouter` connects LangChain to an LLM through OpenRouter.
- `requests` calls the OpenWeatherMap REST API.
- `langsmith` records and displays the agent's execution traces.

In [2]:
# Run this cell once in Google Colab
!pip install -q -U langchain langchain-openrouter langsmith requests

## 2. Add API keys safely

This notebook needs three keys:

- `OPENROUTER_API_KEY` — lets us use the language model.
- `OPENWEATHER_API_KEY` — lets us retrieve live weather data.
- `LANGSMITH_API_KEY` — sends the agent's traces to LangSmith.

`getpass` hides the keys while you type them. Avoid writing API keys directly in a shared notebook.

In [3]:
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass(
    "Enter OPENROUTER_API_KEY: "
)
os.environ["OPENWEATHER_API_KEY"] = getpass.getpass(
    "Enter OPENWEATHER_API_KEY: "
)

# os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
#     "Enter LANGSMITH_API_KEY: "
# )

# # Enable automatic tracing for every LangChain agent invocation.
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "weather-agent-class-demo"

print("✅ API keys loaded")
# print("✅ LangSmith tracing enabled")
# print(f"Project: {os.environ['LANGSMITH_PROJECT']}")

Enter OPENROUTER_API_KEY: ··········
Enter OPENWEATHER_API_KEY: ··········
Enter LANGSMITH_API_KEY: ··········
✅ API keys loaded
✅ LangSmith tracing enabled
Project: weather-agent-class-demo


**LangSmith = CCTV for your agent execution.**

## What does LangSmith tracing add?

LangSmith does not change the agent's answer or give it a new capability. It observes what happens during execution.

For each request, the trace can show:

- The user's input
- The prompt and messages sent to the model
- The model's decision to call `weather_tool`
- The location passed to the tool
- The tool result returned to the model
- The final answer, latency, and token usage when available

Because LangChain agents support LangSmith natively, the three environment variables above are enough to enable automatic tracing.



## 3. What is a weather agent?

A normal LLM does not automatically know the live weather. A **weather agent** combines an LLM with a weather tool.

| Component | Responsibility |
|---|---|
| User | Asks a question in natural language |
| LLM | Understands the request and decides whether to call the tool |
| Weather tool | Calls OpenWeatherMap and returns live data |
| Agent | Coordinates the LLM and tool, then produces the final answer |

The process is:

1. The user asks, `What is the weather in Delhi?`
2. The LLM recognizes that current weather data is required.
3. The agent calls `weather_tool(location="Delhi")`.
4. The tool returns temperature, humidity, wind, and conditions.
5. The LLM converts the tool result into a clear answer.

## 4. Import the libraries

In [4]:
import requests

from langchain.agents import create_agent
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain.tools import tool
from langchain_openrouter import ChatOpenRouter

print("✅ Imports complete")

✅ Imports complete


## 5. Create the weather tool

The `@tool` decorator converts a normal Python function into a tool that the agent can call.

The function's docstring is important: the LLM reads it to understand **when** the tool should be used and **what input** it expects.

In [5]:
@tool
def weather_tool(location: str) -> str:
    """Get the current weather for a city or location.

    Use this tool for questions about current temperature, conditions,
    humidity, wind, rain, or weather in a particular location.
    """
    print(f"WEATHER TOOL CALLED for: {location}")

    api_key = os.environ.get("OPENWEATHER_API_KEY")
    if not api_key:
        return "OPENWEATHER_API_KEY is missing. Please set it first."

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": api_key,
        "units": "metric",
    }

    try:
        response = requests.get(url, params=params, timeout=20)
        print(response)
        if response.status_code != 200:
            try:
                error_message = response.json().get("message", response.text)
            except ValueError:
                error_message = response.text
            return f"Weather API error ({response.status_code}): {error_message}"

        data = response.json()

    except requests.RequestException as error:
        return f"Could not connect to the weather service: {error}"

    city = data.get("name", location)
    country = data.get("sys", {}).get("country", "")
    condition = data.get("weather", [{}])[0].get("description", "unknown")
    temperature = data.get("main", {}).get("temp", "unknown")
    feels_like = data.get("main", {}).get("feels_like", "unknown")
    humidity = data.get("main", {}).get("humidity", "unknown")
    wind_speed = data.get("wind", {}).get("speed", "unknown")

    return (
        f"Current weather in {city}, {country}:\n"
        f"- Condition: {condition}\n"
        f"- Temperature: {temperature}°C\n"
        f"- Feels like: {feels_like}°C\n"
        f"- Humidity: {humidity}%\n"
        f"- Wind speed: {wind_speed} m/s"
    )

print("✅ Weather tool created")

✅ Weather tool created


200 → Success

401 → Invalid/unauthorized API key

404 → Location not found

429 → Too many API requests

### Test the tool directly

Before creating the agent, call the tool directly. This confirms that the API key and weather API are working.

In [6]:
weather_tool.invoke({"location": "Kolkata"})

WEATHER TOOL CALLED for: Kolkata
<Response [200]>


'Current weather in Kolkata, IN:\n- Condition: light rain\n- Temperature: 29.97°C\n- Feels like: 36.97°C\n- Humidity: 84%\n- Wind speed: 5.66 m/s'

## 6. Initialize the LLM

The LLM understands the question, chooses the weather tool when necessary, and writes the final response.

In [7]:
llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_tokens=250,
)

print("✅ LLM initialized")

✅ LLM initialized


In [8]:
llm.invoke("What is the weather in Kolkata?")

AIMessage(content="I don't have real-time data access to provide current weather information. To get the latest weather updates for Kolkata, I recommend checking a reliable weather website or using a weather app.", additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1786860558-sePIgZFGwRDV8SJxE3qK', 'created': 1786860558, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 2.37e-05, 'cost_details': {'upstream_inference_completions_cost': 2.16e-05, 'upstream_inference_prompt_cost': 2.1e-06, 'upstream_inference_cost': 2.37e-05}, 'system_fingerprint': 'fp_369e662417'}, id='lc_run--01a00930-5804-7da3-a342-5c66e67113fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 36, 'total_tokens': 50, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

## 7. Create the weather agent

The agent receives only one tool: `weather_tool`. Its system prompt defines the agent's role and tool-use rules.

In [9]:
system_prompt = """
You are a helpful weather assistant.

Rules:
- Use weather_tool whenever live or current weather data is required.
- If the user does not specify a location and it cannot be understood
  from the conversation, ask for the location.
- Use earlier weather results for follow-up questions when they are
  already available in the conversation.
- If the user explicitly asks for refreshed or latest data, call the tool again.
- Do not invent weather values.
- Keep the final answer clear and beginner-friendly.
- If the user asks for temperature, return ONLY the temperature value
  with °C.
"""

weather_agent = create_agent(
    model=llm,
    tools=[weather_tool],
    system_prompt=system_prompt,
)

print("✅ Weather agent created")

✅ Weather agent created




```
User
 ↓
Weather Agent
 ↓
LLM reads the question
 ↓
"Do I need current weather data?"
 ↓
YES
 ↓
weather_tool(location="Kolkata")
 ↓
OpenWeather API
 ↓
Weather data returned
 ↓
weather_tool formats the data
 ↓
Result goes back to the Agent / LLM
 ↓
LLM creates a natural-language response
 ↓
User



### Test the agent without saved memory

Here we send only one user message. The agent can answer it, but this cell does not save the conversation for the next request.

In [10]:
result = weather_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the wind speed in Delhi?"}
    ]
})
# print(result["messages"])
print(result["messages"][-1].content)

WEATHER TOOL CALLED for: Delhi
<Response [200]>
The wind speed in Delhi is 4.63 m/s.


## 8. Why does the agent need chat-history memory?

Suppose the first question is `What is the weather in Kolkata?` and the next question is `What is the humidity there?`

The word **there** refers to Kolkata. If the second request is sent alone, the agent may not know which location the user means.

In this notebook, memory works by:

1. Saving every returned message in `full_chat_history`.
2. Adding the new user question to that history.
3. Sending the complete conversation back to the agent.
4. Replacing the stored history with the agent's updated message list.

> The model itself is not permanently remembering the conversation. Our Python code stores the messages and sends them again with every request.

## 9. Add full chat-history memory

In [11]:
full_chat_history = []


def ask_weather_agent(question: str) -> str:
    """Ask the weather agent while preserving the full conversation."""
    global full_chat_history

    # Add the new question after all previous conversation messages.
    messages = full_chat_history + [
        {"role": "user", "content": question}
    ]

    # The result contains user, assistant, tool-call, and tool-result messages.
    result = weather_agent.invoke({"messages": messages})

    # Save the complete updated history for the next question.
    full_chat_history = result["messages"]

    answer = full_chat_history[-1].content
    print(f"User: {question}")
    print(f"Assistant: {answer}\n")
    return answer


def clear_chat_history() -> None:
    """Delete all stored conversation messages."""
    global full_chat_history
    full_chat_history = []
    print("✅ Full chat history cleared")





```
System prompt        1,000 tokens
Previous messages   95,000 tokens
New question           500 tokens
Tool results          2,000 tokens
Expected response     3,000 tokens
--------------------------------
Total              101,500 tokens ❌





```
More history
   ↓
More input tokens
   ↓
Higher API cost
   ↓
Slower requests
   ↓
More irrelevant context
   ↓
Potentially worse answers```



## 10. Test the memory-enabled weather agent

Run the questions in order. The second and third questions depend on information from the first answer.

In [12]:
clear_chat_history()

ask_weather_agent("What is the current weather in Kolkata?")
ask_weather_agent("What is the humidity there?")
ask_weather_agent("Based on that weather, should I carry an umbrella?")

# Traces are uploaded in the background. This waits until they are submitted.
# wait_for_all_tracers()
# print("✅ Traces submitted to LangSmith")

✅ Full chat history cleared
WEATHER TOOL CALLED for: Kolkata
<Response [200]>
User: What is the current weather in Kolkata?
Assistant: The current temperature in Kolkata is 29.97°C.

User: What is the humidity there?
Assistant: The humidity in Kolkata is 84%.

User: Based on that weather, should I carry an umbrella?
Assistant: Yes, you should carry an umbrella since there is light rain in Kolkata.

✅ Traces submitted to LangSmith


### View the trace

1. Open [LangSmith](https://smith.langchain.com/).
2. Open the **Tracing** section.
3. Select the project **weather-agent-class-demo**.
4. Open a trace to inspect the agent steps, model calls, weather-tool call, inputs, outputs, latency, and token usage.

Each call to `ask_weather_agent()` creates a new trace. Since full chat history is resent, later traces also show the earlier conversation messages.

## 11. Inspect the stored history

An agent interaction can produce more than two messages because tool calls and tool results are also stored.

In [13]:
fruits = ["Apple", "Banana", "Mango"]

for number, fruit in enumerate(fruits, start=1):
    print(number, fruit)

1 Apple
2 Banana
3 Mango


In [14]:
print(f"Total stored messages: {len(full_chat_history)}\n")

for number, message in enumerate(full_chat_history, start=1):
    message_type = getattr(message, "type", "unknown")
    print(f"{number}. {message_type}: {message.content}\n")

Total stored messages: 8

1. human: What is the current weather in Kolkata?

2. ai: 

3. tool: Current weather in Kolkata, IN:
- Condition: light rain
- Temperature: 29.97°C
- Feels like: 36.97°C
- Humidity: 84%
- Wind speed: 5.66 m/s

4. ai: The current temperature in Kolkata is 29.97°C.

5. human: What is the humidity there?

6. ai: The humidity in Kolkata is 84%.

7. human: Based on that weather, should I carry an umbrella?

8. ai: Yes, you should carry an umbrella since there is light rain in Kolkata.



## 12. Optional interactive chat

Run this cell to continue chatting. Type `exit` to stop or `clear` to erase the stored history.

In [15]:
while True:
    question = input("You: " ).strip()

    if question.lower() == "exit":
        wait_for_all_tracers()
        print("Chat ended.")
        break

    if question.lower() == "clear":
        clear_chat_history()
        continue

    if question:
        ask_weather_agent(question)

You: exit
Chat ended.


## Key takeaways

- The LLM handles language and decides when the weather tool is needed.
- The weather tool retrieves live data from OpenWeatherMap.
- The agent coordinates the LLM and the tool.
- `full_chat_history` gives conversational memory by preserving and resending every message.
- Full history is easy to understand, but it grows over time and increases token usage. In a production application, a window or summary memory strategy may be more economical.
- Setting `LANGSMITH_TRACING=true` enables automatic tracing for the LangChain agent.
- `LANGSMITH_PROJECT` groups the traces under a readable project name.
- LangSmith helps inspect model calls, tool calls, inputs, outputs, latency, and token usage without changing the agent's logic.